In [1]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.experiments.metrics import Loss
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.utils import class_name_to_str
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
from IPython.display import display, Markdown

from workspace.sources.local_datasets.preprocessing.encoders.encoders import RoBERTaEncoder as Encoder
from workspace.sources.models.transformers.bert_based_models import RoBERTa as Model

import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [2]:
experiment_name = f'prefinal-{class_name_to_str(Model.__name__.lower())}-v2'
print('Experiment:', experiment_name)

Experiment: prefinal-ro_ber_ta-v2


In [3]:
def conduct_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid):
    total_runs = len(dataset_hparams_grid) * len(model_hparams_grid)
    for i, dataset_params in enumerate(dataset_hparams_grid):
        for j, model_hparams in enumerate(model_hparams_grid):
            run_number = i * len(model_hparams_grid) + j + 1
            display(Markdown(f'### Run {run_number} of {total_runs}'))
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = Model(train_best_model_metric=Loss,
                           training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            recovery_experiment.run()


In [4]:

# max_tokens_params = [128, 512]
max_tokens_params = [128]

pipelines = []

for max_tokens in max_tokens_params:
    pipelines.extend([PreprocessingPipeline(name='minimal',
                                            iterable=[Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction',
                                            iterable=[NoiseReduction(),
                                                      Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_stemming',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)])
                      ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

# model_hparams_grid = ParameterGrid({'epochs': [10],
#                                     'batch_size': [16],
#                                     'learning_rate': [1e-05, 5e-05],
#                                     'lr_scheduler': ['linear'],
#                                     'optimizer': ['adamw_torch'],
#                                     'weight_decay': [0.0, 1e-3],
#                                     'early_stopping_patience': [3],
#                                     'early_stopping_threshold': [0.01]})

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


### ReCOVery Dataset

In [5]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 02:14:30,089 - Experiment - INFO - MLflow experiment initialized with ID: 407824870238116211
2025-05-16 02:14:30,090 - Experiment - INFO - Model signature: model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 02:14:30,091 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),ro_ber_ta_encoder(t=roberta-base,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:14:30,091 - Experiment - INFO - Run signature: model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),ro_ber_ta_encoder(t=roberta-base,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:14:30,216 - Experiment - INFO - Found existing run with signature model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_t

### Run 2 of 4

2025-05-16 02:14:30,226 - Experiment - INFO - MLflow experiment initialized with ID: 407824870238116211
2025-05-16 02:14:30,227 - Experiment - INFO - Model signature: model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 02:14:30,227 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),noise_reduction(),ro_ber_ta_encoder(t=roberta-base,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:14:30,228 - Experiment - INFO - Run signature: model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),noise_reduction(),ro_ber_ta_encoder(t=roberta-base,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:14:30,347 - Experiment - INFO - Found existing run with signature model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,l

### Run 3 of 4

...

2025-05-16 02:16:03,763 - Experiment - INFO - Test metrics: {'test_loss': 1.4969736337661743, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.7222222222222222, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1716, 'test_samples_per_second': 87.401, 'test_steps_per_second': 5.827, 'test_epoch': 9.0}
2025-05-16 02:16:03,780 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:16:03,782 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:16:03,820 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:16:03,821 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 02:16:03,822 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-35
2025-05-16 02:16:04,303 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.583891749382019, 'eval_accuracy': 0.7333333333333333, 'eval_precisi

2025-05-16 02:16:04,442 - Experiment - INFO - Test metrics: {'test_loss': 1.502776861190796, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.6666666666666667, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1334, 'test_samples_per_second': 112.405, 'test_steps_per_second': 7.494, 'test_epoch': 7.0}
2025-05-16 02:16:04,461 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:16:04,463 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:16:04,503 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:16:04,503 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 02:16:04,504 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-45
2025-05-16 02:16:04,967 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6796987056732178, 'eval_accuracy': 0.7333333333333333, 

2025-05-16 02:16:05,104 - Experiment - INFO - Test metrics: {'test_loss': 1.4969736337661743, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.7222222222222222, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1323, 'test_samples_per_second': 113.354, 'test_steps_per_second': 7.557, 'test_epoch': 9.0}
2025-05-16 02:16:05,122 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:16:05,126 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:16:05,166 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:16:05,167 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 02:16:05,169 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-20
2025-05-16 02:16:05,711 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.9430989027023315, 'eval_accuracy': 0.6, 'eval_precision': 0.6, 'eva

2025-05-16 02:16:05,856 - Experiment - INFO - Test metrics: {'test_loss': 1.165282964706421, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.537037037037037, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1379, 'test_samples_per_second': 108.77, 'test_steps_per_second': 7.251, 'test_epoch': 4.0}
2025-05-16 02:16:05,885 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:16:05,888 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:16:05,940 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:16:05,942 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 02:16:05,942 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-30
2025-05-16 02:16:06,542 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.5153636336326599, 'eval_accuracy': 0.6, 'eval_precision': 0.6363636363636

2025-05-16 02:16:06,679 - Experiment - INFO - Test metrics: {'test_loss': 0.852662980556488, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.6296296296296297, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1328, 'test_samples_per_second': 112.963, 'test_steps_per_second': 7.531, 'test_epoch': 6.0}
2025-05-16 02:16:06,698 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:16:06,701 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:16:06,744 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:16:06,745 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### ISOT Dataset

In [ ]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### CZ Dataset

In [6]:
cz_pipelines = []

for max_tokens in max_tokens_params:
    cz_pipelines.extend([PreprocessingPipeline(name='minimal',
                                               iterable=[Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction',
                                               iterable=[NoiseReduction(),
                                                         Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Lemmatization(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_stemming',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Stemming(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)])
                         ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(cz_pipelines, fraction=0.15)

cz_dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': cz_pipelines})
print('Dataset Hyperparameters Grid Size: ', len(cz_dataset_hparams_grid))
print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(cz_dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


In [7]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          cz_dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 02:16:07,315 - Experiment - INFO - MLflow experiment initialized with ID: 407824870238116211
2025-05-16 02:16:07,316 - Experiment - INFO - Model signature: model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 02:16:07,317 - Experiment - INFO - Dataset signature: dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.15,tt=top),ro_ber_ta_encoder(t=roberta-base,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:16:07,317 - Experiment - INFO - Run signature: model(mn=roberta,ti=roberta-base,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.15,tt=top),ro_ber_ta_encoder(t=roberta-base,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:16:08,531 - Experiment - INFO - Run ID: 607a37c07e7c404ba80bde99c82861a9
2025-05-16 02:16:08,532 - Experiment - INFO - Run name: model(mn=roberta,ti=roberta-ba

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 02:19:39,495 - Experiment - INFO - Test metrics: {'test_loss': 0.4660516679286957, 'test_accuracy': 0.8181818181818182, 'test_precision': 1.0, 'test_recall': 0.7333333333333333, 'test_f1_score': 0.8461538461538461, 'test_roc_auc': 0.9047619047619047, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.26666666666666666, 'test_runtime': 0.2448, 'test_samples_per_second': 89.873, 'test_steps_per_second': 8.17, 'test_epoch': 6.0}
2025-05-16 02:19:39,516 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:19:39,518 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:19:39,574 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:19:39,575 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 02:19:39,575 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-49
2025-05-16 02:19:40,093 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.71096163

2025-05-16 02:19:40,281 - Experiment - INFO - Test metrics: {'test_loss': 0.677480161190033, 'test_accuracy': 0.6818181818181818, 'test_precision': 0.75, 'test_recall': 0.8, 'test_f1_score': 0.7741935483870968, 'test_roc_auc': 0.7714285714285716, 'test_false_positives_rate': 0.5714285714285714, 'test_false_negatives_rate': 0.2, 'test_runtime': 0.1824, 'test_samples_per_second': 120.642, 'test_steps_per_second': 10.967, 'test_epoch': 7.0}
2025-05-16 02:19:40,303 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:19:40,305 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:19:40,368 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:19:40,369 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 02:19:40,370 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-7
2025-05-16 02:19:40,883 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6988145112

C:\Users\nikita.bardatskii\git\bi-bap-2025-bardanik\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2025-05-16 02:19:41,067 - Experiment - INFO - Test metrics: {'test_loss': 0.704379677772522, 'test_accuracy': 0.3181818181818182, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1_score': 0.0, 'test_roc_auc': 0.9523809523809524, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 1.0, 'test_runtime': 0.1797, 'test_samples_per_second': 122.414, 'test_steps_per_second': 11.129, 'test_epoch': 1.0}
2025-05-16 02:19:41,085 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:19:41,087 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:19:41,149 - Experiment - INFO - Successfully saved evaluation data.


2025-05-16 02:19:41,950 - Experiment - INFO - Test metrics: {'test_loss': 0.5804975628852844, 'test_accuracy': 0.6818181818181818, 'test_precision': 0.6818181818181818, 'test_recall': 1.0, 'test_f1_score': 0.8108108108108109, 'test_roc_auc': 0.8857142857142857, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1996, 'test_samples_per_second': 110.241, 'test_steps_per_second': 10.022, 'test_epoch': 4.0}
2025-05-16 02:19:41,977 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:19:41,981 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:19:42,052 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:19:42,054 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 02:19:42,055 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-42
2025-05-16 02:19:42,588 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.5725111961364746, 'eval_a

2025-05-16 02:19:42,780 - Experiment - INFO - Test metrics: {'test_loss': 0.4660516679286957, 'test_accuracy': 0.8181818181818182, 'test_precision': 1.0, 'test_recall': 0.7333333333333333, 'test_f1_score': 0.8461538461538461, 'test_roc_auc': 0.9047619047619047, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.26666666666666666, 'test_runtime': 0.1864, 'test_samples_per_second': 118.029, 'test_steps_per_second': 10.73, 'test_epoch': 6.0}
2025-05-16 02:19:42,800 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:19:42,802 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:19:42,869 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:19:42,870 - Experiment - INFO - Finished model evaluations stage.
